# **RNN과 자연어처리**


---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---



# **1. 텍스트 표현의 역사**:
**n-gram → 토큰화 → BoW·TF-ID**F

| Lab |  주제 | 발전사 위치 |
|---|---|---|
| 1-1 |  바이그램 언어 모델로 "다음 단어" 맞히기 | 1948 Shannon · 1980~90s 통계적 NLP |
| 1-2 | 공백 · 음절 · 형태소 · BPE 토크나이저 비교 | 1990s 형태소 분석 → 2016 BPE |
| 1-3 |  TF-IDF + 로지스틱 회귀 감성 분류 (기준 모델) | 1972 IDF · 2000s 기계학습 NLP |
| 1-4 | Q.영화 리뷰로 배운 모델이 게임 리뷰도 이해할까? | 도메인 이동 문제 |

**사용 데이터셋**
- 한국어 자연어 처리(NLP), 감성 분석(Sentiment Analysis)을 위한 공개 코퍼스
1. NSMC(Naver Sentiment Movie Corpus):
    - NSMC 네이버 영화 리뷰 20만 건 (Park, 2016, CC0)
    — 4차시 LLM 제로샷 실습(영화 리뷰를 긍정/부정으로 분류)에서 사용한 데이터
    - 도메인 편향됨(영화 도메인에 국한되어 있어 일상 텍스트 반영 어려움 있음)
    - https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
2. Steam 게임 리뷰 10만 건 (bab2min/corpus, ):
    - 형태소 분석기(Kiwi) 및 토픽 모델링 도구(Tomoto) 개발 시 벤치마킹 데이터로 활용  
    - Public Domain
    - https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt

In [ ]:
# ▶ 0. 설치 (Colab 기준 약 20초)
%pip install -q kiwipiepy koreanize-matplotlib tokenizers

In [ ]:
# ▶ 0. 데이터 내려받기 — GitHub에서 바로 받는다 (회원가입 불필요)
import os, urllib.request
import pandas as pd

URLS = {
    "ratings_train.txt": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "ratings_test.txt":  "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    "steam.txt":         "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt",
}
for fn, url in URLS.items():
    if not os.path.exists(fn):
        urllib.request.urlretrieve(url, fn)

# NSMC: id \t document \t label (1=긍정, 0=부정)
train = pd.read_csv("ratings_train.txt", sep="\t").dropna()
test  = pd.read_csv("ratings_test.txt",  sep="\t").dropna()
# Steam: label \t review (헤더 없음)
steam = pd.read_csv("steam.txt", sep="\t", header=None, names=["label", "document"], quoting=3).dropna()
print(f"NSMC train {len(train):,} / test {len(test):,} / Steam {len(steam):,}")
print("긍정 비율:", round(train.label.mean(), 3))
train.head()

---
## **1-1. 바이그램 언어 모델** — "진짜" 다음에는 무엇이 올까?
$P(w_t \mid w_{t-1}) = \dfrac{\text{count}(w_{t-1}, w_t)}{\text{count}(w_{t-1})}$ — 최대우도 추정(MLE), 세어서 나누기만 하면 된다.

### **[실습] 바이그램으로 다음 단어 맞추기**

In [ ]:
from collections import Counter, defaultdict

# 1) 공백으로 자르고 문장 시작<s>/끝</s> 기호를 붙인다
sents = [["<s>"] + d.split() + ["</s>"] for d in train.document]

# 2) 앞 단어별로 다음 단어를 센다
bigram = defaultdict(Counter)
for s in sents:
    for w1, w2 in zip(s[:-1], s[1:]):
        bigram[w1][w2] += 1


def next_words(w, k=5):
    total = sum(bigram[w].values())
    return [(nw, round(c / total, 3)) for nw, c in bigram[w].most_common(k)]

for w in ["진짜", "이", "평점", "배우"]:
    print(f"{w:>4} →", next_words(w))

In [ ]:
# 3) 가장 확률 높은 단어를 계속 이어 붙여 문장 만들기 (greedy)
def generate(start="이", n=10):
    out = [start]
    for _ in range(n):
        nxt = bigram[out[-1]].most_common(1)
        if not nxt or nxt[0][0] == "</s>":
            break
        out.append(nxt[0][0])
    return " ".join(out)

for w in ["이", "진짜", "배우들"]:
    print(generate(w))

In [ ]:
# 4) n-gram의 한계: 희소성(sparsity) — 테스트 문장의 바이그램 중 학습에서 한 번도 못 본 비율
test_bi = [(a, b) for d in test.document for a, b in zip(["<s>"] + d.split(), d.split() + ["</s>"])]
unseen = sum(1 for a, b in test_bi if bigram[a][b] == 0) / len(test_bi)
print(f"테스트 바이그램 중 학습 때 못 본 비율: {unseen:.1%}  → 확률 0 문제, 스무딩이 필요한 이유")

### **[Quiz] `generate()`가 같은 말을 반복하는 이유는?**

- [답변]
    
    
    
    .

### **[실습]: 확률에 비례해 <strong>샘플링<strong>하도록 바꿔 보자!**
앞 bigram 출력방법에서 generate()가 같은 말을 계속 반복하는 경우를 처리하기 위한 조치를 취해보세요. (힌트: `random.choices`).

In [ ]:
import random

def generate_sampling(start="이", n=10):
    out = [start]
    for _ in range(n):
        current_word = out[-1]
        candidates = bigram[current_word]

        if not candidates:
            break

        # 후보 단어들과 그 빈도 추출
        words, counts = zip(*candidates.items())

        # 빈도를 가중치(weights)로 활용하여 확률적 샘플링 진행, 복원추출(비복원추출=random.sample())
        nxt_word = random.choices(words, weights=counts, k=1)[0]

        if nxt_word == "</s>":
            break

        out.append(nxt_word)

    return " ".join(out)

# 샘플링 방식 테스트
print("--- 확률 기반 샘플링 결과 ---")
for w in ["이", "진짜", "배우들"]:
    print(f"{w:<4} -> {generate_sampling(w)}")

---



## **1-2. Tokenization**

### **1. 한국어는 어떻게 잘라야 할까?** 공백 · 음절 · 형태소 · BPE

- **BPE(Byte-Pair Encoding): 서브워드 토큰화 알고리즘**
    - 텍스트 내에서 **가장 빈번하게 연속되어 등장하는 문자 쌍(Pair)을 반복적으로 병합**(Merge)하여 서브워드(Subword) 어휘 사전을 구축하는 압축 기반 토큰화 알고리즘
- BPE 기본 동작 원리
    1. 개별 문자(Character/Byte) 단위로 단어를 분절하여 초기 어휘 집합(Vocabulary) 생성
    2. 코퍼스 내 인접한 토큰 쌍의 출현 빈도수(Frequency) 계산
    3. 빈도수가 가장 높은 쌍을 하나의 새로운 토큰으로 병합
    4. 사전 정의된 어휘 사전 크기(Vocab Size) 또는 병합 반복 횟수에 도달할 때까지 반복

### **[실습] 한국어 자르는 방법: 공백 · 음절 · 형태소 · BPE**

In [ ]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

samples = ["이 영화 개꿀잼ㅋㅋ 배우들 연기 미쳤다", "스토리는 노잼인데 영상미는 좋았어요", "재밌었다. 재미있다. 재밌어요!"]

# (1) 공백  (2) 음절(문자)  (3) Kiwi 형태소 분석
for s in samples[:1]:
    print("공백  :", s.split())
    print("음절  :", list(s.replace(" ", "")))
    print("형태소:", [t.form for t in kiwi.tokenize(s)])

In [ ]:
# (4) BPE: NSMC로 직접 학습한 서브워드 토크나이저 (Sennrich et al., 2016)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

bpe = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe.pre_tokenizer = pre_tokenizers.Whitespace()
bpe.train_from_iterator(train.document.tolist(),
                        trainers.BpeTrainer(vocab_size=8000, special_tokens=["[UNK]"]))
for s in samples:
    print("BPE   :", bpe.encode(s).tokens)

### **[실습] 방법 비교**: 공백 · 음절 · 형태소 · BPE
- [주의!] CPU에서 실행 오래 걸릴 수 있음(T4 2분)

In [ ]:
# 네 가지 방식 정량 비교: 어휘 크기 · 테스트 OOV 비율 · 문장당 토큰 수
# OOV(Out-Of-Vocabulary): 미등록 단어
import numpy as np, time

def kiwi_tok(texts):
    return [[t.form for t in r] for r in kiwi.tokenize(texts)]   # 리스트를 넣으면 병렬 처리

t0 = time.time()
tok_train_kiwi = kiwi_tok(train.document.tolist())
tok_test_kiwi  = kiwi_tok(test.document.tolist())
print(f"Kiwi 형태소 분석 {len(train)+len(test):,}문장: {time.time()-t0:.0f}초")

schemes = {
    "공백(어절)": (lambda d: d.split(), None),
    "음절(문자)": (lambda d: list(d.replace(" ", "")), None),
    "Kiwi 형태소": (None, (tok_train_kiwi, tok_test_kiwi)),
    "BPE(8k)": (lambda d: bpe.encode(d).tokens, None),
}
rows = []
for name, (fn, pre) in schemes.items():
    tr, te = pre if pre else ([fn(d) for d in train.document], [fn(d) for d in test.document])
    vocab = Counter(w for s in tr for w in s)
    v5 = {w for w, c in vocab.items() if c >= 5}            # 5번 이상 나온 토큰만 어휘로 사용
    te_tokens = [w for s in te for w in s]
    oov = sum(1 for w in te_tokens if w not in v5) / len(te_tokens)
    rows.append([name, len(vocab), len(v5), f"{oov:.1%}", round(np.mean([len(s) for s in te]), 1)])
tok_table = pd.DataFrame(rows, columns=["방식", "전체 어휘", "어휘(빈도≥5)", "테스트 OOV", "문장당 토큰"])
tok_table

**관찰**: 공백 기준은 "재밌었다/재밌어요"를 모두 다른 단어로 본다 → 어휘 폭증 + OOV. 형태소·BPE는 어휘를 줄이고 OOV를 낮춘다.

### 🟠 kiwipiepy 소개
C++로 작성된 지능형 한국어 형태소 분석기 **Kiwi**(Korean Intelligent Word Identifier)의 공식 Python 래퍼(바인딩)
- 주요 기능
    - 형태소 분석 및 품사 태깅 (tokenize)
    - 멀티스레드 배치 분석 (tokenize 내 num_workers 옵션)
    - 띄어쓰기 오류 및 오타 자동 보정
    - 사용자 사전(동적 추가/추출) 및 불용어 관리 (Stopwords)
    - https://bab2min.github.io/kiwipiepy/v0.24.1/kr/

- **[예제1] 기본 토큰화 및 금융 복합명사 등록**
> *기본* 형태소 분석과 함께 신조어/금융 용어가 분해되는 것을 방지하기 위해 사용자 사전에 등록하는 예제

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 1. 미등록 상태 분석
finance_text = "토스뱅크에서 비대면 신용대출을 신청했다."
tokens_before = kiwi.tokenize(finance_text)
print("[사전 추가 전]")
print([f"{t.form}/{t.tag}" for t in tokens_before])

# 2. 사용자 단어 사전 추가 (고유명사 NNP, 가중치 점수 부여)
kiwi.add_user_word("토스뱅크", "NNP", 5.0)
kiwi.add_user_word("신용대출", "NNG", 5.0)

# 3. 사전 추가 후 재분석
tokens_after = kiwi.tokenize(finance_text)
print("\n[사전 추가 후]")
print([f"{t.form}/{t.tag}" for t in tokens_after])

- **[예제2] 불용어 필터링 및 띄어쓰기/오타 교정 파이프라인**
> 의료 도메인에서 환자 상담 텍스트에 흔히 발생하는 띄어쓰기 오류를 보정하고 임상적으로 유의미한 체언·용언만 추출하는 예제

In [ ]:
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords

kiwi = Kiwi(typos="basic")  # 오타 자동 교정 기능 활성화
stopwords = Stopwords()

# 의료 상담 도메인 텍스트 (띄어쓰기 누락 및 일상 불용어 포함)
medical_text = "환자가머리가너무아프고열이나서진료를받았습니다."

# 1. 띄어쓰기 공간 분할 보정 수행
space_corrected = kiwi.space(medical_text)
print(f"띄어쓰기 교정 결과: {space_corrected}")

# 2. 형태소 분석 및 불용어 제거 (명사 및 동사/형용사 어간 추출)
tokens = kiwi.tokenize(space_corrected, stopwords=stopwords)
meaningful_words = [
    t.form for t in tokens if t.tag.startswith("NN") or t.tag.startswith("V")
]

print(f"추출된 핵심 키워드: {meaningful_words}")

---
## **1-3. TF-IDF + 로지스틱 회귀**: **기준 모델(baseline)**
$\text{tfidf}(t,d) = \text{tf}(t,d) \times \log\dfrac{1+N}{1+\text{df}(t)} + 1$ (scikit-learn 기본식, smooth_idf=True)

### **[실습] 기준 모델(baseline) 만들기**
토큰 단위(어절·음절·형태소)별로 TF-IDF + 로지스틱 회귀를 학습하고 테스트 정확도를 비교하세요.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def run_tfidf(name, Xtr_text, Xte_text, **vec_kw):
    t0 = time.time()
    vec = TfidfVectorizer(min_df=3, sublinear_tf=True, **vec_kw)
    Xtr, Xte = vec.fit_transform(Xtr_text), vec.transform(Xte_text)   # fit은 train에만! (2차시 데이터 누수)
    clf = LogisticRegression(C=2.0, max_iter=1000).fit(Xtr, train.label)
    acc = accuracy_score(test.label, clf.predict(Xte))
    print(f"{name:<22} 특성 {Xtr.shape[1]:>7,}개  정확도 {acc:.4f}  ({time.time()-t0:.0f}초)")
    return vec, clf, acc

join = lambda toks: [" ".join(t) for t in toks]
results = {}
_, _, results["BoW-TF-IDF 어절 1-gram"] = run_tfidf("✅ 어절 1-gram", train.document, test.document)
_, _, results["TF-IDF 어절 1~2gram"]    = run_tfidf("✅ 어절 1~2gram", train.document, test.document, ngram_range=(1, 2))
_, _, results["TF-IDF 음절 1~3gram"]    = run_tfidf("✅ 음절 1~3gram", train.document, test.document, analyzer="char_wb", ngram_range=(1, 3))
vec_k, clf_k, results["TF-IDF 형태소 1~2gram"] = run_tfidf("✅ 형태소 1~2gram", join(tok_train_kiwi), join(tok_test_kiwi),
                                                          ngram_range=(1, 2), token_pattern=None, tokenizer=str.split)

### [실습] 모델이 배운 단어와 BoW 한계 확인하기

In [ ]:
# 모델이 배운 "긍정 단어 / 부정 단어" — 로지스틱 회귀 가중치 상위
names = np.array(vec_k.get_feature_names_out())
coef = clf_k.coef_[0]
print("긍정 Top 15:", list(names[np.argsort(coef)[-15:][::-1]]))
print("부정 Top 15:", list(names[np.argsort(coef)[:15]]))

In [ ]:
# BoW의 한계: 어순을 모른다 — 같은 단어, 다른 뜻
for s in ["재미 없는 건 아니고 진짜 최고",
          "진짜 최고 아니고 재미 없는 건",
          "처음엔 지루했는데 끝까지 보니 인생 영화",
          "처음엔 재밌었는데 끝까지 보니 시간 낭비",
          "배우 연기는 좋은데 스토리가 별로"]:
    x = vec_k.transform([" ".join(t.form for t in kiwi.tokenize(s))])
    print(f"{s:<22} → 긍정 확률 {clf_k.predict_proba(x)[0,1]:.3f}")

---
## **1-4. 영화 리뷰로 배운 모델, 게임 리뷰도 알아들을까?**
학습 분포와 다른 데이터(도메인 이동, domain shift)에서 성능이 어떻게 변하는지 확인한다.

### **[실습] 영화 리뷰로 배운 모델을 이용하여 게임 리뷰 판별하기**

In [ ]:
# 1. 전체 스팀 데이터 중 20,000개를 무작위로 추출합니다.
steam_eval = steam.sample(20000, random_state=42)

# 2. 추출한 스팀 리뷰(document)들을 Kiwi 형태소 분석기로 토큰화합니다.
steam_documents = steam_eval.document.tolist()
tok_steam = kiwi_tok(steam_documents)

# 3. 토큰화된 리스트들을 다시 공백으로 구분된 문자열로 합쳐줍니다. (TF-IDF 입력 형식 맞추기)
# 예: ['영화', '재밌', '다'] -> '영화 재밌 다'
steam_joined_texts = join(tok_steam)

# 4. 영화 리뷰로 미리 학습해둔 TF-IDF 벡터라이저(vec_k)로 스팀 리뷰를 수치 벡터로 변환합니다.
X_steam = vec_k.transform(steam_joined_texts)

# 5. 영화 리뷰로 미리 학습해둔 분류기(clf_k)를 사용하여 변환된 스팀 데이터를 예측합니다.
predictions = clf_k.predict(X_steam)

# 6. 실제 정답 레이블(steam_eval.label)과 예측값(predictions)을 비교하여 정확도를 구합니다.
y_true = steam_eval.label
acc_steam = accuracy_score(y_true, predictions)

# 7. 네이버 영화 리뷰(NSMC) 테스트 결과와 스팀 게임 리뷰 테스트 결과를 비교하여 출력합니다.
nsmc_accuracy = results['TF-IDF 형태소 1~2gram']
print(f"NSMC 영화 리뷰 테스트 정확도: {nsmc_accuracy:.4f}")
print(f"Steam 게임 리뷰 테스트 정확도: {acc_steam:.4f}")

In [ ]:
# 결과 저장 (3교시 RNN과 비교할 기준선)
import json
results["Steam(도메인 이동)"] = acc_steam
json.dump(results, open("baseline_tfidf.json", "w"), ensure_ascii=False, indent=1)
tok_table.to_csv("tokenizer_compare.csv", index=False)
results

### [미션] 다른 도메인 데이터 섞어서 학습시킨 모델의 정확도 확인하기
NSMC 데이터에 Steam 데이터 1,000건을 학습에 섞으면 정확도는 얼마나 회복될까?